In [ ]:
# Lab6 참고 노트북 — 프로젝트 루트는 cwd와 무관하게 탐색
from pathlib import Path
import sys

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "project_paths.py").is_file():
        sys.path.insert(0, str(_p))
        break
from project_paths import project_root, data_dir, artifacts_lab6

ROOT = project_root()
DATA_DIR = data_dir()
ART_LAB6 = artifacts_lab6()

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass, InitVar, field, asdict

### DATASET and PARAMETER

@dataclass(frozen=True)
class HyperParameter:
    batch_size: int
    num_epochs: int
    learning_rate: float

@dataclass
class MNIST:
    root: InitVar[str]
    param: InitVar[HyperParameter]

    train: DataLoader = field(init=False)
    test:  DataLoader = field(init=False)
    total: DataLoader = field(init=False)

    def __post_init__(self, root, param):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        train_set = datasets.MNIST(
            root=root,
            train=True,
            download=True,
            transform=transform,
        )
        test_set = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=transform,
        )
        self.train = DataLoader(
            dataset=train_set,
            batch_size=param.batch_size, shuffle=True
        )
        self.test = DataLoader(
            dataset=test_set,
            batch_size=param.batch_size, shuffle=False
        )
        self.total = DataLoader(
            dataset=train_set+test_set,
            batch_size=param.batch_size, shuffle=False
        )

In [8]:
### FIXED-POINT EMULATION: ap_fixed<16, 7>
FIXED_W = 16
FIXED_I = 7
FIXED_F = FIXED_W - FIXED_I

FIXED_SCALE = 2 ** FIXED_F
FIXED_MIN = -(2 ** (FIXED_I - 1))
FIXED_MAX = (2 ** (FIXED_I - 1)) - (1 / FIXED_SCALE)

def fixed_point_round(x):
    """
    Emulate signed ap_fixed<16, 7>.

    W = 16
    I = 7
    F = 9
    Range = [-64, 64)
    Step size = 2^-9
    """
    x = torch.clamp(x, FIXED_MIN, FIXED_MAX)
    x = torch.round(x * FIXED_SCALE) / FIXED_SCALE
    return x

class FixedPointQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return fixed_point_round(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-Through Estimator (STE)
        return grad_output

class QuantizedLinear(nn.Linear):
    def forward(self, x):
        q_weight = FixedPointQuantize.apply(self.weight)
        q_bias = FixedPointQuantize.apply(self.bias) if self.bias is not None else None
        return F.linear(x, q_weight, q_bias)

### NEURAL NETWORK DEFINITION
class MNISTSLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = QuantizedLinear(784, 10, bias=True)

    def forward(self, x):
        x = x.view(x.size(0), -1)

        # Input image fixed-point emulation
        x = FixedPointQuantize.apply(x)

        # Weight/bias fixed-point emulation is applied inside QuantizedLinear
        x = self.fc(x)
        return x

In [9]:
import os
from pathlib import Path
from datetime import datetime

### TRAIN / EVALUATION ROUTINE

@dataclass
class Routine:
    device: object
    criterion: object
    optimizer: object

    def _routine(self, model, loader):
        num_hit, running_loss = 0, 0.0
        
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            loss = self.criterion(outputs, labels)

            if torch.is_grad_enabled():
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            num_hit += (predicted==labels).sum().item()
            running_loss += loss.item()
            
        return num_hit / len(loader.dataset), \
               running_loss / len(loader)

    def train(self, model, loader):
        model.train()
        return self._routine(model, loader)

    def test(self, model, loader):
        model.eval()
        with torch.no_grad():
            return self._routine(model, loader)
            
@dataclass
class Milestone:
    parent: str | Path
    model_name: str
    ext: str
    now: str = field(init=False)

    def __post_init__(self):
        self.now = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
        self.parent = Path(self.parent)

        if not self.parent.exists():
            self.parent.mkdir()

    def __iter__(self):
        files = self.parent.glob(f"{self.model_name}-*.{self.ext}")
        return iter(sorted(files, key=lambda f: f.name, reverse=True))

    def save(self, **kwargs):
        torch.save(kwargs, self.parent / f"{self.model_name}-{self.now}.{self.ext}")

In [10]:
### TRAIN / EVALUATION ROUTINE
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
param = HyperParameter(
    batch_size=128,
    num_epochs=20,
    learning_rate=1e-3
)
mnist = MNIST(str(DATA_DIR), param)
model = MNISTSLP().to(device)

criterion =  nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=param.learning_rate)
routine = Routine(device, criterion, optimizer)

In [12]:
milestone = Milestone(str(ROOT / "milestones_qat"), type(model).__name__, 'pth')
print(f"Train Starts at: {milestone.now}")
print(f"Target Model: {milestone.model_name}")
print('')
print(f"| {'EPOCH': ^5} | {'TRAIN': ^33} | {'TEST': ^33} |")
print(f"| {'     ': ^5} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} |")

checkpoint = None
for epoch in range(param.num_epochs):
    train_acc, train_loss = routine.train(model, mnist.train)
    test_acc, test_loss = routine.test(model, mnist.test)

    print(f"| {epoch: ^5} "
          f"| {train_acc: ^15.2%} | {train_loss: ^15.5} "
          f"| {test_acc: ^15.2%} | {test_loss: ^15.5} |", end='')

    if (checkpoint is None) or (checkpoint['accuracy'] < test_acc):
        checkpoint = dict(
            epoch=epoch,
            accuracy=test_acc,
            param=asdict(param),
            model=model.state_dict(),
            optim=optimizer.state_dict(),
        )
        milestone.save(**checkpoint)
        print(' *', end='')
    print('')

Train Starts at: 2026-05-12-09-39-19
Target Model: MNISTSLP

| EPOCH |               TRAIN               |               TEST                |
|       |    ACCURACY     |      LOSS       |    ACCURACY     |      LOSS       |
|   0   |     85.28%      |     0.54016     |     90.45%      |     0.34221     | *
|   1   |     90.33%      |     0.33855     |     91.16%      |     0.31841     | *
|   2   |     91.00%      |     0.31436     |     91.60%      |     0.29324     | *
|   3   |     91.17%      |     0.3046      |     90.64%      |     0.30545     |
|   4   |     91.58%      |      0.296      |     92.02%      |     0.27647     | *
|   5   |     91.73%      |     0.29121     |     91.98%      |     0.27969     |
|   6   |     91.83%      |     0.28776     |     91.89%      |     0.27898     |
|   7   |     91.99%      |     0.28318     |     91.80%      |     0.28429     |
|   8   |     91.97%      |     0.28314     |     91.93%      |     0.28641     |
|   9   |     92.14%      |  

In [13]:
model_name = 'MNISTSLP'
test_ms = Milestone(milestone.parent, model_name, milestone.ext)
test_model = globals()[model_name]()
test_model.eval()

with torch.no_grad():
    for ms_file in test_ms:
        checkpoint = torch.load(ms_file)
    
        test_model.load_state_dict(checkpoint['model'])
        test_model.to(device)
        accuracy, _ = routine.test(test_model, mnist.total)
        print(ms_file.stem, ':\t', f"{accuracy: .2%}")

MNISTSLP-2026-05-12-09-39-19 :	  92.82%
MNISTSLP-2026-05-12-09-36-06 :	  92.61%
MNISTSLP-2026-05-12-09-33-15 :	  92.80%
